# 第十五课｜电脑怎样和现场可编程门阵列（Field-Programmable Gate Array, FPGA）说话？

有了开发板后，下一步先建立最小通信路径：

> **电脑里的软件怎样写一个值给 FPGA，再把 FPGA 中的结果读回来？**

这里把运行控制程序的一侧叫**主机（host）**，把由 bitstream 配置的硬件区域叫**可编程逻辑（programmable logic, PL）**。

主要新概念：**host 与 PL 是两个执行域。**

## 1. 概念账本

**已经知道：** development board、clock、reset、I/O、bitstream。

**今天学习：**
- **主机（host）**：运行控制程序的电脑或处理器一侧；
- **中央处理器（Central Processing Unit, CPU）**；
- **片上系统（System on Chip, SoC）**；
- **可编程逻辑（programmable logic, PL）**：由 bitstream 配置的逻辑区域。

**只预告：** 后面会学习常用的标准片上通信协议，但今天不学它的通道细节。

## 2. 两个执行域

```mermaid
flowchart LR
  HOST["host software / CPU"] -->|write command| IF["platform control path"]
  IF --> PL["programmable logic register / engine"]
  PL -->|result| IF
  IF -->|readback| HOST
```

host 软件按指令执行；PL 电路按 clock 与信号关系持续工作。一次寄存器写入跨过的是硬件通信路径，不只是 Python 变量赋值。

## 3. 为什么先做 loopback？

最小 loopback 把问题缩成四件事：host 能否送出值、PL 是否收到并保存/处理、host 能否读回、顺序与 state 是否正确。

这四步没验证前，不应同时调试神经元、FIFO、DDR 和连接组。

## 4. Run：模拟命令顺序

下面只是**通信语义模型**，不是实际总线实现。先预测两次 read 会读到什么。

In [ ]:
pl_register = 0
commands = [
    ("write", 7),
    ("add", 5),
    ("read", None),
    ("add", -2),
    ("read", None),
]

readbacks = []
for op, value in commands:
    if op == "write":
        pl_register = value
    elif op == "add":
        pl_register += value
    elif op == "read":
        readbacks.append(pl_register)

print("host readbacks:", readbacks)
print("final PL register:", pl_register)


## 5. Observe

`write` 建立 PL register 的 state；`add` 在这个 state 上继续修改；`read` 只观察当前值。

重点是 host command 与 PL state 之间的顺序，不是 Python 语法。

## 6. SoC FPGA 为什么方便？

SoC 把 CPU 系统与 programmable logic 紧密放在同一平台。CPU 适合文件、网络、实验控制；PL 适合确定、并行的数据路径。

但“在同一颗芯片或封装里”不等于使用同一种执行语义。

## 7. Try It

在第一次 `add` 之前插入一个 `("read", None)`。先预测新的 readback 顺序。

## 8. 作业

[第 15 课作业：模拟 host ↔ programmable logic 往返](../../exercises/zh/15_host_talks_to_fpga.ipynb)

## 9. AI Task

让 AI 为“host write → PL update → host read”画 Mermaid 图，并标出哪些动作属于软件、哪些属于同步逻辑。检查它是否把 Python 函数调用与真实硬件通信操作混为一谈。

## 10. Human Check

解释 host/CPU 与 PL 各自执行什么；为什么 loopback 比直接上神经网络更适合作为第一条通信测试；readback 为什么同时验证返回路径；为什么具体总线协议的细节可以推迟学习。

## 11. Engineering Handoff

对应 `RMD-012B / RMD-013`：先完成 host ↔ FPGA minimal loopback，再迁移平台 3 已验证的小网络。

## 12. 项目追踪 Project Trace

- Lesson: `LSN-015`
- Mapping: `RMD-012B / RMD-013`
- Minimal proof: host write → PL update/state → host read
- Boundary: specific bus-protocol details deferred

## 13. Exit Ticket

你能说明 host program 与 FPGA logic 为什么是两个执行域，并能按顺序追踪一次最小 roundtrip。